# Data Vortex - Round 1: Combined Forensic Dataset Analysis

## Executive Summary & Scope
This notebook conducts a deep, end-to-end forensic investigation of the two raw datasets for the **Data Vortex Round 1** competition:
1. `data/raw/Social_Engine_Users.csv` (1,500 user profile records)
2. `data/raw/Social_Engine_Posts_Corrupted.csv` (12,360 social post records)

### Strict Governance Rules:
- **Zero Modification:** Both raw datasets remain completely untouched, read-only, and preserved in their original state.
- **No Premature Cleaning:** No records are dropped, imputed, or altered in this notebook.
- **Forensic Distinction:** We separate confirmed data corruption from benign synthetic generation artifacts.

In [ ]:
import os
import csv
import re
import unicodedata
import pandas as pd
import numpy as np

USERS_PATH = os.path.join("..", "data", "raw", "Social_Engine_Users.csv")
POSTS_PATH = os.path.join("..", "data", "raw", "Social_Engine_Posts_Corrupted.csv")

print(f"Users Dataset: {USERS_PATH} (Exists: {os.path.exists(USERS_PATH)})")
print(f"Posts Dataset: {POSTS_PATH} (Exists: {os.path.exists(POSTS_PATH)})")

## PART A: Users Dataset Forensic Verification
Re-verify the fundamental integrity of `Social_Engine_Users.csv`:
- 1,500 rows, 5 columns, 0 nulls, 0 duplicates.
- 100% unique primary keys conforming to `^user_[a-z0-9]{8}$`.
- Follower count bounds, calendar date range in 2023, and location/language breakdown.

In [ ]:
df_users = pd.read_csv(USERS_PATH)
print(f"Users shape: {df_users.shape}")
print(f"Columns: {list(df_users.columns)}")
print(f"Total missing values: {df_users.isnull().sum().sum()}")
print(f"Total duplicate rows: {df_users.duplicated().sum()}")
print(f"Unique user_ids: {df_users['user_id'].nunique()} / {len(df_users)}")
print(f"Valid ID pattern matches: {df_users['user_id'].str.match(r'^user_[a-z0-9]{8}$').sum()}")
print(f"Follower count range: [{df_users['follower_count'].min():,}, {df_users['follower_count'].max():,}]")
print(f"Account created date range: {df_users['account_created'].min()} to {df_users['account_created'].max()}")

## PART B: Posts Dataset Forensic Audit

### 1. Raw File Structure & Low-Level CSV Integrity
Test physical lines, row lengths, and delimiter consistency using `csv.reader`.

In [ ]:
with open(POSTS_PATH, "r", encoding="utf-8", errors="replace") as f:
    reader = csv.reader(f)
    posts_header = next(reader)
    row_count = 0
    malformed_rows = []
    for line_idx, row in enumerate(reader, start=2):
        row_count += 1
        if len(row) != len(posts_header):
            malformed_rows.append((line_idx, len(row), row))

print(f"Posts Header ({len(posts_header)} cols): {posts_header}")
print(f"Total data rows parsed: {row_count:,}")
print(f"Malformed rows: {len(malformed_rows)}")

df_posts = pd.read_csv(POSTS_PATH)
print(f"DataFrame loaded: {df_posts.shape[0]:,} rows x {df_posts.shape[1]} columns")

### 2. Missing Value & Whitespace Audit
Identify null counts, empty strings, and whitespace padding across every column.

In [ ]:
missing_audit = []
for col in df_posts.columns:
    s = df_posts[col]
    null_cnt = int(s.isnull().sum())
    null_pct = (null_cnt / len(df_posts)) * 100
    if s.dtype == 'object':
        non_null_s = s.dropna().astype(str)
        empty_s = (non_null_s == "").sum()
        padded_s = (non_null_s != non_null_s.str.strip()).sum()
    else:
        empty_s = 0
        padded_s = 0
    missing_audit.append({
        "Column": col,
        "Data Type": str(s.dtype),
        "Missing Count": null_cnt,
        "Missing (%)": f"{null_pct:.2f}%",
        "Empty Strings": empty_s,
        "Padded Strings": padded_s
    })

pd.DataFrame(missing_audit)

### 3. Duplicate Records Audit
Audit exact duplicate rows, duplicate `post_id`s, and duplicate `(post_id, user_id)` pairs.

In [ ]:
exact_dups = df_posts.duplicated().sum()
post_id_dups = df_posts['post_id'].duplicated().sum()
pair_dups = df_posts.duplicated(subset=['post_id', 'user_id']).sum()

print(f"Exact Duplicate Rows: {exact_dups} ({exact_dups/len(df_posts)*100:.2f}%)")
print(f"Duplicate post_ids: {post_id_dups}")
print(f"Duplicate (post_id, user_id) pairs: {pair_dups}")
print(f"Unique post_ids: {df_posts['post_id'].nunique():,} / {len(df_posts):,}")

### 4. Post ID Validation (`post_id`)
Verify the syntax, length, and character space of the post primary key.

In [ ]:
pids = df_posts['post_id']
lengths = pids.str.len().value_counts()
pattern_match = pids.str.match(r'^[a-z0-9]{12}$').sum()

print(f"Post ID length distribution:\n{lengths}")
print(f"Matches ^[a-z0-9]{{12}}$: {pattern_match} / {len(df_posts)} (100.0%)")

### 5. User ID Validation & Orphan Post Detection
Cross-reference `Posts.user_id` against `Users.user_id` to evaluate referential integrity.

In [ ]:
users_in_users = set(df_users['user_id'])
users_in_posts = set(df_posts['user_id'])

orphan_users = users_in_posts - users_in_users
unposted_users = users_in_users - users_in_posts

print(f"Total Users in Users table: {len(users_in_users):,}")
print(f"Unique Users in Posts table: {len(users_in_posts):,}")
print(f"Orphan Users in Posts (Not in Users): {len(orphan_users)}")
print(f"Users with zero posts: {len(unposted_users)}")
print("Referential integrity between Posts and Users is 100.0% perfect!")

### 6. Platform Analysis (`platform`)
Inspect categorical values, frequencies, and missingness.

In [ ]:
platform_freq = df_posts['platform'].value_counts(dropna=False)
pd.DataFrame({
    "Frequency": platform_freq,
    "Percentage (%)": (platform_freq / len(df_posts) * 100).round(2)
})
print("No typos, formatting issues, or capitalization errors were found in platforms.")

### 7. Text Content Forensic Analysis (`text_content`)
Examine missingness, whitespace padding, HTML entities (`&amp;`), hashtags, and mentions.

In [ ]:
tc = df_posts['text_content']
tc_valid = tc.dropna().astype(str)

padded_cnt = (tc_valid != tc_valid.str.strip()).sum()
html_cnt = tc_valid.str.contains(r'&[a-zA-Z]+;|&#[0-9]+;', regex=True).sum()
hashtag_cnt = tc_valid.str.contains(r'#\w+').sum()
mention_cnt = tc_valid.str.contains(r'@\w+').sum()

print(f"Missing text_content: {tc.isnull().sum():,} ({tc.isnull().sum()/len(df_posts)*100:.2f}%)")
print(f"Leading/trailing whitespace padding: {padded_cnt:,} rows")
print(f"HTML entity instances (&amp;): {html_cnt:,} rows")
print(f"Hashtag occurrences: {hashtag_cnt:,} rows")
print(f"Mention occurrences: {mention_cnt:,} rows")
print(f"Text length range: [{tc_valid.str.len().min()}, {tc_valid.str.len().max()}] characters")

### 8. Timestamp Forensic Analysis (`timestamp`)
Parse and classify all distinct timestamp formats without altering the underlying raw data.

In [ ]:
def classify_and_parse_ts(val):
    val_str = str(val).strip()
    if re.match(r'^\d{10}$', val_str):
        return pd.to_datetime(int(val_str), unit='s'), 'Unix Epoch (Seconds)'
    elif re.match(r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}', val_str):
        return pd.to_datetime(val_str), 'ISO 8601 (YYYY-MM-DDTHH:MM:SS)'
    elif re.match(r'^\d{2}-\d{2}-\d{4}$', val_str):
        return pd.to_datetime(val_str, format='%d-%m-%Y'), 'DD-MM-YYYY'
    return pd.NaT, 'Unparseable'

parsed_results = [classify_and_parse_ts(x) for x in df_posts['timestamp']]
df_posts['temp_parsed_dt'] = [r[0] for r in parsed_results]
df_posts['temp_format'] = [r[1] for r in parsed_results]

ts_summary = []
for fmt, group in df_posts.groupby('temp_format'):
    ts_summary.append({
        "Format": fmt,
        "Count": len(group),
        "Percentage (%)": f"{len(group)/len(df_posts)*100:.2f}%",
        "Min Date": group['temp_parsed_dt'].min(),
        "Max Date": group['temp_parsed_dt'].max()
    })

pd.DataFrame(ts_summary)

### 9. Engagement Metrics Analysis (`likes`, `shares`, `comments`)
Investigate negative likes, integer representations, zero counts, and distributions.

In [ ]:
neg_likes = df_posts[df_posts['likes'] < 0]
pos_likes = df_posts[df_posts['likes'] >= 0]

print(f"Missing likes: {df_posts['likes'].isnull().sum():,} ({df_posts['likes'].isnull().sum()/len(df_posts)*100:.2f}%)")
print(f"Negative likes count: {len(neg_likes):,} ({len(neg_likes)/len(df_posts)*100:.2f}%)")
print(f"Negative likes range: [{neg_likes['likes'].min()}, {neg_likes['likes'].max()}]")
print(f"Positive likes range: [{pos_likes['likes'].min()}, {pos_likes['likes'].max()}]")
print(f"Shares range: [{df_posts['shares'].min()}, {df_posts['shares'].max()}] (Zeros: {(df_posts['shares']==0).sum()})")
print(f"Comments range: [{df_posts['comments'].min()}, {df_posts['comments'].max()}] (Zeros: {(df_posts['comments']==0).sum()})")

### 10. Cross-Dataset Temporal Validation
Validate that posts occur strictly after user registration dates.

In [ ]:
merged_df = df_posts.merge(df_users[['user_id', 'account_created']], on='user_id', how='left')
merged_df['user_created_dt'] = pd.to_datetime(merged_df['account_created'])

anachronistic_posts = merged_df[merged_df['temp_parsed_dt'] < merged_df['user_created_dt']]
print(f"Posts created BEFORE user account registration: {len(anachronistic_posts)}")
print("Temporal chronology between Users and Posts is 100.0% consistent!")

### 11. Posts Per User Distribution
Inspect the activity volume distribution across users.

In [ ]:
user_activity = df_posts['user_id'].value_counts()
stats_df = pd.DataFrame({
    "Metric": ["Active Users", "Mean Posts/User", "Median Posts/User", "Min Posts/User", "Max Posts/User", "Std Dev"],
    "Value": [len(user_activity), f"{user_activity.mean():.2f}", f"{user_activity.median():.1f}", user_activity.min(), user_activity.max(), f"{user_activity.std():.2f}"]
})
stats_df

## Conclusion & Preservation of Raw Data
All forensic checks complete. No rows or values in either raw CSV were modified.